# Calculate depth to water table from the surface from ATS simulation outputs

##### The following script calculates the water table depth from ATS-simulated subsurface pressure.
##### It employs a bottom-up approach to identify the last saturated cell starting from the bottom soil layers. In other words, the pressure of the last saturated cell, indexed from the bottom (where the bottom-most subsurface cell has an index of 0), is used to calculate the water table depth.

#### Formula

##### Water_Table_depth = z_surface_coords - z_cell_coords - [Pressure_cell - Pressure_atm] / [water_density * g]

##### where:
- `z_surface_coords`, `z_cell_coords` are the z coordinates [m] of the surface and the identified last saturated subsurface cell, respectively.
- `Pressure_cell` is the corresponding cell's subsurface pressure [Pascals].


### User Inputs ::: 


In [ ]:
# Define the output model directory, whre ats_vis data (.h5) are located
model_dir = './run2-transient/Oak-0/'

# Define the Parameters AND verify with the XML
rho = 997 # density of water, kg m^-3
g = 9.80665 # gravity, m s^-2
patm = 101325 # atmopsheric pressure, Pascals

### Loading necessary pacakges

In [ ]:
import modvis.ats_xdmf as xdmf
import numpy as np
import time
import random
import pandas
import os
import h5py

### function to estimate WTD based on pressure 

In [ ]:
def get_ats_wtd_pressurebased(pressure_subsurface,visfile_surface, visfile_subsurface):
    # visfile_subsurface.centroids.shape -> (n_surface, 14, 3); 14 is soil layers, bottom-top
    iz_coord = visfile_subsurface.centroids[:,:,-1]
    
    ### Find Equivalent Surface and Subsurface ID based on cell centroids
    # take some care on the rounding of coordinates, can cause errors
    surface_centroids_rounded = np.round(visfile_surface.centroids[:, :2], 4)
    subsurface_centroids_rounded = np.round(visfile_subsurface.centroids[:, 0, :2], 4)
    # [remarks] prepare to perform a pairwise compare [n_surface,1,2] with [1,n_surface,2] -> returns [n_surface, n_surface]
    surface_centroids_expanded = surface_centroids_rounded[:, np.newaxis, :]
    subsurface_centroids_expanded = subsurface_centroids_rounded[np.newaxis, :, :]
    matches = np.all(surface_centroids_expanded == subsurface_centroids_expanded, axis=-1)
    surface_indices, subsurface_indices = np.nonzero(matches)
    surface_subsurface_IDs = np.column_stack((surface_indices, subsurface_indices))
    assert surface_subsurface_IDs[:,1].shape == visfile_surface.centroids[:,1].shape, f"Shape mismatch: change the round precision in the surface/subsurface centroid coordinates"
    
    ### Estimate WTD based on pressure
    # pressure head
    ih = (pressure_subsurface - patm) / (rho * g) # dim=(time, n_surface, 14)
    mask = ih > 0
    first_false_idx = (~mask).argmax(axis=-1) # dim=(time, n_surface)
    max_len = mask.shape[-1]
    indices = np.arange(max_len)
    mask2 = indices < first_false_idx[..., np.newaxis]
    all_true_rows = (first_false_idx == 0)
    mask2[all_true_rows] = True
    sat_idx = mask2[:, :, ::-1].argmax(axis=-1)
    sat_idx = mask2.shape[-1] - 1 - sat_idx
    sat_idx[~mask2.any(axis=-1)] = 0
    # WTD elevation
    # [remark] iH_rev=part1+part2; part1 is the relative distance between water table and the first saturated subsurface cell
    iH_rev = ih[np.arange(ih.shape[0])[:, None], np.arange(ih.shape[1]), sat_idx] + iz_coord[np.arange(ih.shape[1]), sat_idx] 
    first_depth_to_centroid = visfile_surface.centroids[surface_subsurface_IDs[0][0], -1] - visfile_subsurface.centroids[surface_subsurface_IDs[0][1], -1, -1]
    # WTD (from surface)
    head_pressure_based = -(iz_coord[:,-1]+first_depth_to_centroid - iH_rev) #[added by Yi]
    wtd_pressure_based = np.maximum(iz_coord[:,-1]+first_depth_to_centroid - iH_rev, 0) 
    assert pressure_subsurface[:,:,1].shape == wtd_pressure_based.shape, f"Shape mismatch: Error in WTD calculation"
    
    ### Re-arrange WTD in accordance to surface IDs
    # As pressure obtained from subsurface does not follow surface ID, re-arrange this:
    wtd_pressure_rearranged = wtd_pressure_based[:, subsurface_indices]
    head_rearranged = head_pressure_based[:, subsurface_indices]
    
    return wtd_pressure_rearranged, surface_subsurface_IDs, head_rearranged

#### 3D case: Gather inputs and get WTD

- took 550s to process the cell below

In [ ]:
# subsurface
start = time.time()
visfile_subsurface = xdmf.VisFile(model_dir, domain=None, load_mesh=True, columnar=True, ats_version=1.4, model_time_unit='d')
end = time.time()
print(f"Time cost for subsurface VisFile: {end - start:.6f} seconds")

# surface
start = time.time()
visfile_surface = xdmf.VisFile(model_dir, domain='surface', load_mesh=True, ats_version=1.4, model_time_unit='d')
end = time.time()
print(f"Time cost for surface VisFile: {end - start:.6f} seconds")

# subsurface pressure (time, xy-space and soil columns)
start = time.time()
pressure_subsurface = visfile_subsurface.getArray('pressure')
end = time.time()
print(f"Time cost for getting pressure_subsurface: {end - start:.6f} seconds")

# get the WTD (based on surface ATS ID)
start = time.time()
wtd_pressure_rearranged, surface_subsurface_IDs, head_pressure_based = get_ats_wtd_pressurebased(pressure_subsurface=pressure_subsurface,
                                                                            visfile_surface=visfile_surface, visfile_subsurface= visfile_subsurface)
end = time.time()
print(f"Time cost for get_ats_wtd_pressurebased: {end - start:.6f} seconds")


In [ ]:
wtd_pressure_rearranged

In [ ]:
wtd_pressure_rearranged.shape # first dim is time, second dim is space

In [ ]:
surface_subsurface_IDs

#### Yi's operation to extract head at the starting and end points of a 2D transect.

In [ ]:
surface_x_coord = visfile_surface.centroids[:,0]
surface_y_coord = visfile_surface.centroids[:,1]

In [ ]:
print(surface_x_coord)
print(surface_x_coord.shape)
print(surface_y_coord)
print(surface_y_coord.shape)

In [ ]:
#[input] from 1-full_workflow_OakCreek.ipynb
start_coords, end_coords = (-1511015.74507677, 640547.2830956738), (-1511677.2648839361, 640389.829786001)

In [ ]:
# Compute Euclidean distances
dist_start = np.sqrt((surface_x_coord - start_coords[0])**2 + (surface_y_coord - start_coords[1])**2)
dist_end = np.sqrt((surface_x_coord - end_coords[0])**2 + (surface_y_coord - end_coords[1])**2)

# Get the index of the closest point
start_index = np.argmin(dist_start)
end_index = np.argmin(dist_end)

# Get the minimum distances
min_dist_start = dist_start[start_index]
min_dist_end = dist_end[end_index]

# Print results
print(f"Index for start_coords: {start_index}, Minimum distance: {min_dist_start}")
print(f"Index for end_coords: {end_index}, Minimum distance: {min_dist_end}")

In [ ]:
# extract time from model_dir/water_balance_170300020307.csv
# from xml, I expect times=11224 to 16059
# 11224 = 274+365*30 --> 2010.10.1?
# 16059 = 364+365*43 --> 2023.12.31?
df = pandas.read_csv(os.path.join(model_dir, 'water_balance_170300020307.csv'), comment='#')
time = df['time [d]'].values

In [ ]:
print(time)

In [ ]:
output_hdf5_file = os.path.join(".", "bc_startend_raw.h5")

# Extract data based on computed indices
startpt_head = head_rearranged[:, start_index]
endpt_head = head_rearranged[:, end_index]

# Write to HDF5
with h5py.File(output_hdf5_file, "w") as hdf:
    hdf.create_dataset("Time", data=time)
    hdf.create_dataset("startpt_head", data=startpt_head)
    hdf.create_dataset("endpt_head", data=endpt_head)

print(f"Data successfully written to {output_hdf5_file}")

#### Yi's debug log to understand the get_ats_wtd_presrebased() above

In [ ]:
print(pressure_subsurface.shape)
print(pressure_subsurface[0,0,:])

In [ ]:
iz_coord = visfile_subsurface.centroids[:,:,-1]
#print(iz_coord)
print(iz_coord.shape)

In [ ]:
surface_centroids_rounded = np.round(visfile_surface.centroids[:, :2], 4)
print(visfile_surface.centroids.shape)
print(surface_centroids_rounded.shape)
surface_centroids_expanded = surface_centroids_rounded[:, np.newaxis, :]
print(surface_centroids_expanded.shape)

subsurface_centroids_rounded = np.round(visfile_subsurface.centroids[:, 0, :2], 4)
print(visfile_subsurface.centroids.shape)
print(subsurface_centroids_rounded.shape)
subsurface_centroids_expanded = subsurface_centroids_rounded[np.newaxis, :, :]
print(subsurface_centroids_expanded.shape)

In [ ]:
matches = np.all(surface_centroids_expanded == subsurface_centroids_expanded, axis=-1)
print(matches.shape)

In [ ]:
surface_indices, subsurface_indices = np.nonzero(matches)
surface_subsurface_IDs = np.column_stack((surface_indices, subsurface_indices))

In [ ]:
### Estimate WTD based on pressure
# pressure head
ih = (pressure_subsurface - patm) / (rho * g)
mask = ih > 0
first_false_idx = (~mask).argmax(axis=-1)
max_len = mask.shape[-1]
indices = np.arange(max_len)
mask2 = indices < first_false_idx[..., np.newaxis]
all_true_rows = (first_false_idx == 0)
mask2[all_true_rows] = True
sat_idx = mask2[:, :, ::-1].argmax(axis=-1)
sat_idx = mask2.shape[-1] - 1 - sat_idx
sat_idx[~mask2.any(axis=-1)] = 0
# WTD elevation
iH_rev = ih[np.arange(ih.shape[0])[:, None], np.arange(ih.shape[1]), sat_idx] + iz_coord[np.arange(ih.shape[1]), sat_idx] 
first_depth_to_centroid = visfile_surface.centroids[surface_subsurface_IDs[0][0], -1] - visfile_subsurface.centroids[surface_subsurface_IDs[0][1], -1, -1]
# WTD (from surface)
head_pressure_based = -(iz_coord[:,-1]+first_depth_to_centroid - iH_rev) #[added by Yi]
wtd_pressure_based = np.maximum(iz_coord[:,-1]+first_depth_to_centroid - iH_rev, 0) 
assert pressure_subsurface[:,:,1].shape == wtd_pressure_based.shape, f"Shape mismatch: Error in WTD calculation"

### Re-arrange WTD in accordance to surface IDs
# As pressure obtained from subsurface does not follow surface ID, re-arrange this:
wtd_pressure_rearranged = wtd_pressure_based[:, subsurface_indices]
head_rearranged = head_pressure_based[:, subsurface_indices]

In [ ]:
print(ih.shape)
print(mask.shape)
print(first_false_idx.shape)

In [ ]:
tmp_t, tmp_n = 1000, 1000
print(ih[tmp_t, tmp_n])
print(mask[tmp_t,tmp_n,:])
print(first_false_idx[tmp_t, tmp_n])
print(mask2[tmp_t,tmp_n])
print(sat_idx.shape)
print(sat_idx[tmp_t, tmp_n])

print(iH_rev[tmp_t, tmp_n])
print(ih[tmp_t, tmp_n][sat_idx[tmp_t, tmp_n]])
print(iz_coord[tmp_n][sat_idx[tmp_t, tmp_n]])
print(head_pressure_based[tmp_t, tmp_n])
print(wtd_pressure_based[tmp_t, tmp_n])

In [ ]:
#tmp_t, tmp_n = random.randint(0, 4836-1), random.randint(0, 24219-1)
#print([tmp_t, tmp_n])
tmp_t, tmp_n = 0, 0

print(ih[tmp_t, tmp_n])
print(mask[tmp_t,tmp_n,:])
print(first_false_idx[tmp_t, tmp_n])
print(mask2[tmp_t,tmp_n])
print(sat_idx.shape)
print(sat_idx[tmp_t, tmp_n])

print(iH_rev[tmp_t, tmp_n])
print(ih[tmp_t, tmp_n][sat_idx[tmp_t, tmp_n]])
print(iz_coord[tmp_n][sat_idx[tmp_t, tmp_n]])
print(head_pressure_based[tmp_t, tmp_n])
print(wtd_pressure_based[tmp_t, tmp_n])

In [ ]:
print(first_depth_to_centroid)

In [ ]:
print(surface_subsurface_IDs.shape)

In [ ]:
head_pressure_based.shape

In [ ]:
tmp_indices = np.where(head_pressure_based > 0)
print(tmp_indices)
print(tmp_indices[0].shape)
print(tmp_indices[0].shape[0]/(head_pressure_based.shape[0]*head_pressure_based.shape[1])) # percent of element with surface water

In [ ]:
max_index = np.unravel_index(np.argmax(head_pressure_based), head_pressure_based.shape)
print(max_index)
print(head_pressure_based[max_index])

#### 2D case: Gather inputs and get WTD

In [ ]:
# subsurface
start = time.time()
visfile_subsurface = xdmf.VisFile(directory=model_dir, prefix='vis_subsurface',domain=None, load_mesh=True, columnar=True, ats_version=1.4, model_time_unit='d')
end = time.time()
print(f"Time cost for subsurface VisFile: {end - start:.6f} seconds")

# surface
start = time.time()
visfile_surface = xdmf.VisFile(model_dir, domain='surface', prefix ='vis', load_mesh=True, ats_version=1.4, model_time_unit='d')
end = time.time()
print(f"Time cost for surface VisFile: {end - start:.6f} seconds")

# subsurface pressure (time, xy-space and soil columns)
start = time.time()
pressure_subsurface = visfile_subsurface.getArray('pressure')
end = time.time()
print(f"Time cost for getting pressure_subsurface: {end - start:.6f} seconds")

# get the WTD (based on surface ATS ID)
start = time.time()
wtd_pressure_rearranged, surface_subsurface_IDs = get_ats_wtd_pressurebased(pressure_subsurface=pressure_subsurface,
                                                                            visfile_surface=visfile_surface, visfile_subsurface= visfile_subsurface)
end = time.time()
print(f"Time cost for get_ats_wtd_pressurebased: {end - start:.6f} seconds")

In [ ]:
wtd_pressure_rearranged

In [ ]:
wtd_pressure_rearranged.shape